# Solución 11: Tareas de la presentación Runge-Kutta

Origen: `02. DifferentialEquations/Runge-Kutta/runge_kutta.pdf`

Se resuelven los **4 problemas** asignados en clase (RK4 + búsqueda de raíces).

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# IMPLEMENTACIÓN GENÉRICA DE RK4
# ============================================================
def rk4(f, t0, y0, h, t_final):
    """RK4 clásico (Runge). Devuelve (t_arr, y_arr)."""
    t, y = t0, y0
    ts, ys = [t], [y]
    while round(t, 10) < round(t_final, 10):
        k1 = f(t, y)
        k2 = f(t + h/2, y + h*k1/2)
        k3 = f(t + h/2, y + h*k2/2)
        k4 = f(t + h,   y + h*k3)
        y  = y + h/6 * (k1 + 2*k2 + 2*k3 + k4)
        t  = t + h
        ts.append(t); ys.append(y)
    return np.array(ts), np.array(ys)


def get_y_at(f, t0, y0, h, t_target):
    """Evalúa RK4 en t_target con paso fraccionario al final."""
    t, y = t0, y0
    while t + h <= t_target:
        k1 = f(t, y); k2 = f(t+h/2, y+h*k1/2)
        k3 = f(t+h/2, y+h*k2/2); k4 = f(t+h, y+h*k3)
        y += h/6*(k1+2*k2+2*k3+k4); t += h
    if t < t_target:
        hf = t_target - t
        k1 = f(t,y); k2 = f(t+hf/2,y+hf*k1/2)
        k3 = f(t+hf/2,y+hf*k2/2); k4 = f(t+hf,y+hf*k3)
        y += hf/6*(k1+2*k2+2*k3+k4)
    return y

print("Utilidades RK4 cargadas.")

---
## Problema 1 — Circuito RC: tiempo de media carga

$$\frac{dq}{dt} = \frac{V - q/C}{R}, \quad q(0)=0$$

Parámetros: $V=10\,\text{V}$, $R=1000\,\Omega$, $C=10^{-3}\,\text{F}$, $h=0.01\,\text{s}$, span $[0, 5RC]$.

Encontrar $t_m$ tal que $q(t_m) = Q_{\max}/2 = CV/2$ usando el **método de la secante**.

In [ ]:
V, R, C = 10.0, 1000.0, 1e-3
h_rc = 0.01
t_end_rc = 5 * R * C
Q_max = C * V
Q_half = Q_max / 2

f_rc = lambda t, q: (V - q/C) / R

t_rc, q_rc = rk4(f_rc, 0, 0, h_rc, t_end_rc)

# Solución analítica: q(t) = CV(1 - e^{-t/RC})
t_m_exacto = -R*C * np.log(0.5)
print(f"t_m analítico : {t_m_exacto:.6f} s")

# Método de la secante sobre RK4
def phi_rc(t):
    return get_y_at(f_rc, 0, 0, h_rc, t) - Q_half

# Encontrar intervalo
for i in range(1, len(t_rc)):
    if q_rc[i-1] < Q_half <= q_rc[i]:
        t_a_rc, t_b_rc = t_rc[i-1], t_rc[i]
        break

# Secante
xa, xb = t_a_rc, t_b_rc
for _ in range(20):
    fa, fb = phi_rc(xa), phi_rc(xb)
    xc = xb - fb*(xb-xa)/(fb-fa)
    if abs(xc-xb) < 1e-10: break
    xa, xb = xb, xc

print(f"t_m secante   : {xb:.6f} s")
print(f"Error         : {abs(xb - t_m_exacto):.2e} s")

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(t_rc, q_rc*1000, 'royalblue', lw=2, label='q(t) RK4')
ax.axhline(Q_half*1000, color='crimson', ls='--', label=f'Q_max/2 = {Q_half*1000:.2f} mC')
ax.axvline(xb, color='green', ls='--', label=f't_m = {xb:.4f} s')
ax.set_xlabel('t [s]'); ax.set_ylabel('q [mC]')
ax.set_title('Problema 1 — Circuito RC (RK4 + Secante)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Problema 2 — Transferencia radiativa: profundidad óptica de media intensidad

$$\frac{dI}{d\tau} = -I + S, \quad I(0)=1,\; S=0.3$$

Encontrar $\tau_m$ tal que $I(\tau_m) = I_0/2 = 0.5$ usando **bisección**.

In [ ]:
S_rad = 0.3
I0    = 1.0
h_rad = 0.01
t_end_rad = 5.0
I_half = I0 / 2

f_rad = lambda tau, I: -I + S_rad

tau_arr, I_arr = rk4(f_rad, 0, I0, h_rad, t_end_rad)

# Analítica: I(τ) = (I0 - S)e^{-τ} + S
tau_m_exacto = -np.log((I_half - S_rad)/(I0 - S_rad))
print(f"τ_m analítico : {tau_m_exacto:.6f}")

# Bisección
def phi_rad(tau):
    return get_y_at(f_rad, 0, I0, h_rad, tau) - I_half

for i in range(1, len(tau_arr)):
    if I_arr[i-1] > I_half >= I_arr[i]:
        tla, tlb = tau_arr[i-1], tau_arr[i]; break

for _ in range(60):
    tm = (tla+tlb)/2
    if abs(tlb-tla)/2 < 1e-10: break
    if phi_rad(tla)*phi_rad(tm) < 0: tlb = tm
    else: tla = tm

print(f"τ_m bisección : {tm:.6f}")
print(f"Error         : {abs(tm - tau_m_exacto):.2e}")

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(tau_arr, I_arr, 'royalblue', lw=2, label='I(τ) RK4')
ax.plot(tau_arr, (I0-S_rad)*np.exp(-tau_arr)+S_rad, 'k--', lw=1.5, alpha=0.7, label='Analítica')
ax.axhline(I_half, color='crimson', ls='--', label=f'I₀/2 = {I_half}')
ax.axvline(tm, color='green', ls='--', label=f'τ_m = {tm:.4f}')
ax.set_xlabel('τ (profundidad óptica)'); ax.set_ylabel('I(τ)')
ax.set_title('Problema 2 — Transferencia radiativa (RK4 + Bisección)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Problema 3 — Fulguración solar: tiempo de media energía

$$\frac{dE}{dt} = -\alpha E^n, \quad E(0)=1,\; \alpha=0.5,\; n=1.5$$

Encontrar $t_m$ tal que $E(t_m) = E_0/2$ usando **Newton-Raphson**.

In [ ]:
alpha_e, n_e = 0.5, 1.5
E0 = 1.0
h_e = 0.01
t_end_e = 10.0
E_half = E0 / 2

f_energy = lambda t, E: -alpha_e * E**n_e

t_e, E_e = rk4(f_energy, 0, E0, h_e, t_end_e)

# Analítica para n≠1: E(t) = [E0^(1-n) + α(n-1)t]^{1/(1-n)}
def E_analitica(t):
    return (E0**(1-n_e) + alpha_e*(n_e-1)*t)**(1/(1-n_e))

# t_m exacto resolviendo E_analitica = 0.5
t_m_e_ex = (E_half**(1-n_e) - E0**(1-n_e)) / (alpha_e*(n_e-1))
print(f"t_m analítico : {t_m_e_ex:.6f} s")

# Newton-Raphson
def phi_e(t):    return get_y_at(f_energy, 0, E0, h_e, t) - E_half
def dphi_e(t):   # derivada numérica
    dt = 1e-6
    return (phi_e(t+dt) - phi_e(t-dt)) / (2*dt)

# Punto de partida
for i in range(1, len(t_e)):
    if E_e[i-1] > E_half >= E_e[i]: t_nr = (t_e[i-1]+t_e[i])/2; break

for _ in range(20):
    ft, dft = phi_e(t_nr), dphi_e(t_nr)
    t_nr1 = t_nr - ft/dft
    if abs(t_nr1-t_nr) < 1e-10: break
    t_nr = t_nr1

print(f"t_m N-R       : {t_nr:.6f} s")
print(f"Error         : {abs(t_nr - t_m_e_ex):.2e}")

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(t_e, E_e, 'royalblue', lw=2, label='E(t) RK4')
ax.plot(t_e, E_analitica(t_e), 'k--', lw=1.5, alpha=0.7, label='Analítica')
ax.axhline(E_half, color='crimson', ls='--', label='E₀/2 = 0.5')
ax.axvline(t_nr, color='green', ls='--', label=f't_m = {t_nr:.4f} s')
ax.set_xlabel('t [s]'); ax.set_ylabel('E(t)')
ax.set_title('Problema 3 — Fulguración solar (RK4 + Newton-Raphson)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Problema 4 — Burbuja ascendente: tiempo al 99% de velocidad terminal

$$\frac{dv}{dt} = g\frac{\rho_{liq}-\rho_{air}}{\rho_{air}} - \frac{9\mu}{2\rho_{air}r^2}v$$

Encontrar $t_m$ tal que $v(t_m) = 0.99\,v_t$ usando **bisección**.

*(Este problema coincide con el resuelto en `Tareas 11 Abril.ipynb`, aquí se presenta de forma limpia y documentada.)*

In [ ]:
r_b, mu_b = 1e-3, 1e-3
rho_liq, rho_air, g_b = 1000.0, 1.2, 9.81
h_b = 1e-4
t_end_b = 0.5

A_b = g_b * (rho_liq - rho_air) / rho_air
B_b = 9 * mu_b / (2 * rho_air * r_b**2)
v_terminal = A_b / B_b
v_target   = 0.99 * v_terminal

print(f"Velocidad terminal    : {v_terminal:.5f} m/s")
print(f"Objetivo (99% v_t)    : {v_target:.5f} m/s")

f_bub = lambda t, v: A_b - B_b*v

t_b, v_b = rk4(f_bub, 0, 0, h_b, t_end_b)

# Encontrar intervalo para bisección
for i in range(1, len(t_b)):
    if v_b[i-1] < v_target <= v_b[i]:
        tla_b, tlb_b = t_b[i-1], t_b[i]; break

print(f"Cruce RK4 en [{tla_b:.5f}, {tlb_b:.5f}] s")

# Bisección
def phi_b(t): return get_y_at(f_bub, 0, 0, h_b, t) - v_target

tla, tlb = tla_b, tlb_b
for _ in range(60):
    tm_b = (tla+tlb)/2
    if abs(tlb-tla)/2 < 1e-10: break
    if phi_b(tla)*phi_b(tm_b) < 0: tlb = tm_b
    else: tla = tm_b

# Analítica: v(t) = v_t(1 - e^{-B·t})
t_m_b_ex = -np.log(1 - v_target/v_terminal) / B_b
print(f"t_m bisección : {tm_b:.6f} s")
print(f"t_m analítico : {t_m_b_ex:.6f} s")
print(f"Error         : {abs(tm_b - t_m_b_ex):.2e} s")

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(t_b, v_b, 'royalblue', lw=2, label='v(t) RK4')
ax.axhline(v_terminal, color='gray', ls='-', alpha=0.5, label=f'v_t = {v_terminal:.4f} m/s')
ax.axhline(v_target, color='crimson', ls='--', label=f'0.99 v_t = {v_target:.4f} m/s')
ax.axvline(tm_b, color='green', ls='--', label=f't_m = {tm_b:.5f} s')
ax.set_xlim(0, tm_b*4); ax.set_ylim(0, v_terminal*1.05)
ax.set_xlabel('t [s]'); ax.set_ylabel('v [m/s]')
ax.set_title('Problema 4 — Burbuja ascendente (RK4 + Bisección)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Resumen de resultados

| Problema | ODE | Método raíces | Resultado |
|----------|-----|---------------|-----------|
| 1. Circuito RC | $dq/dt=(V-q/C)/R$ | Secante | $t_m = RC\ln 2 \approx 0.6931$ s |
| 2. Transferencia radiativa | $dI/d\tau = -I+S$ | Bisección | $\tau_m \approx 0.847$ |
| 3. Fulguración solar | $dE/dt = -\alpha E^n$ | Newton-Raphson | $t_m \approx 0.828$ s |
| 4. Burbuja ascendente | $dv/dt = A-Bv$ | Bisección | $t_m \approx 0.00123$ s |